In [ ]:
# Create conda environment for DINOv3
conda create -n dinov3-tutorial python=3.9
conda activate dinov3-tutorial

# Install PyTorch for DINOv3
pip install torch torchvision torchaudio

# Install HuggingFace transformers for DINOv3
pip install transformers datasets accelerate

# Additional dependencies for DINOv3 tutorial
pip install opencv-python pillow matplotlib numpyCopy

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import requests

# Load DINOv3 model and processor from HuggingFace
processor = AutoImageProcessor.from_pretrained('facebook/dinov3-base')
model = AutoModel.from_pretrained('facebook/dinov3-base')

# Load example image for DINOv3 processing
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

# Process image with DINOv3
inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)

# Extract DINOv3 features
last_hidden_states = outputs.last_hidden_state
print(f"DINOv3 feature shape: {last_hidden_states.shape}")
# Output: DINOv3 feature shape: torch.Size([1, 257, 768])Copy

In [ ]:
# Complete DINOv3 HuggingFace implementation
from transformers import Dinov3Model, Dinov3ImageProcessor
import torch.nn.functional as F

class DINOv3FeatureExtractor:
    def __init__(self, model_name="facebook/dinov3-large"):
        """Initialize DINOv3 with HuggingFace integration"""
        self.processor = Dinov3ImageProcessor.from_pretrained(model_name)
        self.model = Dinov3Model.from_pretrained(model_name)
        self.model.eval()
    
    def extract_features(self, images, return_cls_token=True):
        """Extract features using DINOv3 HuggingFace model"""
        inputs = self.processor(images=images, return_tensors="pt")
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            
        if return_cls_token:
            # Return CLS token for global features
            return outputs.last_hidden_state[:, 0]
        else:
            # Return all patch tokens for dense features
            return outputs.last_hidden_state[:, 1:]
    
    def get_patch_features(self, images, layer_idx=-1):
        """Get DINOv3 patch features from specific layer"""
        inputs = self.processor(images=images, return_tensors="pt")
        
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)
            
        # Extract features from specific layer
        hidden_states = outputs.hidden_states[layer_idx]
        return hidden_states[:, 1:]  # Exclude CLS token

# Initialize DINOv3 feature extractor
dinov3_extractor = DINOv3FeatureExtractor("facebook/dinov3-base")

# Extract global features
global_features = dinov3_extractor.extract_features([image])
print(f"DINOv3 global features: {global_features.shape}")

# Extract dense patch features for segmentation
patch_features = dinov3_extractor.get_patch_features([image])
print(f"DINOv3 patch features: {patch_features.shape}")Copy

In [ ]:
# Production DINOv3 FastAPI service
from fastapi import FastAPI, UploadFile, File, HTTPException
from typing import List
import torch
import asyncio
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import io

app = FastAPI(title="DINOv3 Production API")

class DINOv3ProductionService:
    def __init__(self):
        """Production-optimized DINOv3 service"""
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load optimized DINOv3 model
        self.model = self._load_optimized_model()
        self.processor = AutoImageProcessor.from_pretrained('facebook/dinov3-base')
        
        # Thread pool for CPU preprocessing
        self.executor = ThreadPoolExecutor(max_workers=4)
    
    def _load_optimized_model(self):
        """Load and optimize DINOv3 for production"""
        model = AutoModel.from_pretrained('facebook/dinov3-base')
        model = model.to(self.device)
        model.eval()
        
        # Enable inference optimizations
        if hasattr(torch.jit, 'optimize_for_inference'):
            model = torch.jit.optimize_for_inference(torch.jit.script(model))
        
        return model
    
    async def extract_features_async(self, images: List[Image.Image]):
        """Async DINOv3 feature extraction"""
        loop = asyncio.get_event_loop()
        
        # Preprocess images in thread pool
        preprocessing_tasks = [
            loop.run_in_executor(
                self.executor, 
                lambda img: self.processor(images=img, return_tensors="pt")
            ) for img in images
        ]
        
        processed_inputs = await asyncio.gather(*preprocessing_tasks)
        
        # Batch inference on GPU
        batch_inputs = torch.cat([inp['pixel_values'] for inp in processed_inputs])
        batch_inputs = batch_inputs.to(self.device)
        
        with torch.no_grad():
            features = self.model(pixel_values=batch_inputs)
            
        return features.last_hidden_state.cpu().numpy()

# Initialize production service
dinov3_service = DINOv3ProductionService()

@app.post("/extract-features/")
async def extract_features(files: List[UploadFile] = File(...)):
    """Extract DINOv3 features from uploaded images"""
    try:
        # Load images
        images = []
        for file in files:
            content = await file.read()
            image = Image.open(io.BytesIO(content)).convert('RGB')
            images.append(image)
        
        # Extract features
        features = await dinov3_service.extract_features_async(images)
        
        return {
            "features": features.tolist(),
            "shape": list(features.shape),
            "model": "DINOv3-Base",
            "status": "success"
        }
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "model": "DINOv3 Production Service"}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)Copy